# X-VC — преобразование голоса

Этот блокнот запускает официальный X-VC с простым Gradio-интерфейсом.

Перед запуском в Colab выберите среду выполнения с GPU. Для первого запуска понадобятся несколько гигабайт загрузок: сам checkpoint X-VC около 5 ГБ, плюс вспомогательные модели.

Запустите первую ячейку один раз. Затем запустите вторую. Она запускает Gradio в фоне, ждёт публичную ссылку и завершается. Когда интерфейс будет готов, ссылка появится отдельной строкой и отдельной ссылкой «Открыть X-VC».


In [ ]:
import os, subprocess
assert subprocess.run(['nvidia-smi'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0, 'Включите GPU в настройках среды выполнения Colab.'
subprocess.run(['rm', '-rf', '/content/audio-restoration-colab'], check=True)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'agent/x-vc-colab', 'https://github.com/egor125552/audio-restoration-colab.git', '/content/audio-restoration-colab'], check=True)
subprocess.run(['bash', '/content/audio-restoration-colab/x_vc_colab/install_xvc.sh', '/content/x-vc'], check=True)
print('X-VC установлена. Теперь запускайте следующую ячейку.')


In [ ]:
import os, re, signal, subprocess, time
from pathlib import Path
from IPython.display import Markdown, display

log_path = Path('/content/xvc-gradio.log')
pid_path = Path('/content/xvc-gradio.pid')

if pid_path.exists():
    try:
        old_pid = int(pid_path.read_text().strip())
        os.kill(old_pid, signal.SIGTERM)
        time.sleep(1)
    except (ValueError, ProcessLookupError, PermissionError):
        pass

subprocess.run(['pkill', '-f', '/content/audio-restoration-colab/x_vc_colab/app.py'], check=False)
log_handle = log_path.open('w', encoding='utf-8')
proc = subprocess.Popen(
    ['/content/x-vc/.venv/bin/python', '-u', '/content/audio-restoration-colab/x_vc_colab/app.py', '--share', '--port', '7860'],
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
log_handle.close()
pid_path.write_text(str(proc.pid), encoding='utf-8')
print('Запускаю интерфейс X-VC. Жду публичную ссылку...')

pattern = re.compile(r'https://[a-zA-Z0-9.-]+\.gradio\.live')
public_url = None
deadline = time.time() + 90
while time.time() < deadline:
    text = log_path.read_text(encoding='utf-8', errors='replace') if log_path.exists() else ''
    match = pattern.search(text)
    if match:
        public_url = match.group(0)
        break
    if proc.poll() is not None:
        break
    time.sleep(1)

if public_url:
    print('X-VC интерфейс готов.')
    print(public_url)
    display(Markdown(f'[Открыть X-VC]({public_url})'))
else:
    tail = log_path.read_text(encoding='utf-8', errors='replace')[-6000:] if log_path.exists() else 'Лог не создан.'
    raise RuntimeError('Gradio не выдал публичную ссылку. Последние строки запуска:\n' + tail)


In [ ]:
import os, signal
from pathlib import Path
pid_path = Path('/content/xvc-gradio.pid')
if pid_path.exists():
    try:
        os.kill(int(pid_path.read_text().strip()), signal.SIGTERM)
        print('Интерфейс X-VC остановлен.')
    except ProcessLookupError:
        print('Интерфейс уже остановлен.')
else:
    print('Запущенный интерфейс не найден.')
